# NutriWise — Exploratory Data Analysis & Model Evaluation
Run after `python scripts/download_datasets.py` and `python scripts/train_models.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA_DIR  = Path('../backend/data')
MODEL_DIR = Path('../backend/models')
PLOT_DIR  = Path('../backend/plots')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('✅ Imports ready')

## 1. Dataset Overview

In [ ]:
nhanes = pd.read_csv(DATA_DIR / 'nhanes_synthetic.csv')
foods  = pd.read_csv(DATA_DIR / 'food_nutrients.csv') if (DATA_DIR / 'food_nutrients.csv').exists() else pd.DataFrame()
ifct   = pd.read_csv(DATA_DIR / 'ifct_indian_foods.csv')

print(f'NHANES rows  : {len(nhanes):,}')
print(f'USDA foods   : {len(foods):,}')
print(f'IFCT (India) : {len(ifct):,}')
nhanes.describe().round(2)

## 2. TDEE Distribution by Goal & Activity

In [ ]:
fig = px.box(
    nhanes, x='activity_level', y='tdee', color='goal',
    category_orders={'activity_level': ['sedentary','light','moderate','very','extreme']},
    color_discrete_map={'loss':'#E76F51','muscle':'#2D6A4F','maintain':'#4A90D9'},
    title='TDEE Distribution by Activity Level and Goal',
    labels={'tdee': 'TDEE (kcal)', 'activity_level': 'Activity Level'}
)
fig.update_layout(height=450)
fig.show()

## 3. Macro Distribution (Protein/Carbs/Fat by Goal)

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['Protein (g)', 'Carbs (g)', 'Fat (g)'])
colors = {'loss': '#E76F51', 'muscle': '#2D6A4F', 'maintain': '#4A90D9'}

for goal, color in colors.items():
    sub = nhanes[nhanes['goal'] == goal]
    for i, col in enumerate(['protein_g', 'carbs_g', 'fat_g'], 1):
        fig.add_trace(go.Violin(
            y=sub[col], name=goal, line_color=color,
            showlegend=(i == 1), legendgroup=goal
        ), row=1, col=i)

fig.update_layout(height=400, title='Macro Distribution by Goal', violinmode='overlay')
fig.show()

## 4. BMI vs TDEE Scatter (colored by goal)

In [ ]:
sample = nhanes.sample(3000, random_state=42)
fig = px.scatter(
    sample, x='bmi', y='tdee', color='goal', size='weight_kg',
    hover_data=['age', 'sex', 'activity_level'],
    color_discrete_map={'loss':'#E76F51','muscle':'#2D6A4F','maintain':'#4A90D9'},
    title='BMI vs TDEE (bubble size = weight)',
    opacity=0.5
)
fig.update_layout(height=500)
fig.show()

## 5. Indian Foods — Top High-Protein (per 100g)

In [ ]:
top_protein = ifct.nlargest(15, 'protein_g')
fig = px.bar(
    top_protein, x='protein_g', y='name', orientation='h',
    color='category',
    title='Top 15 Indian Foods by Protein Content (per 100g)',
    labels={'protein_g': 'Protein (g/100g)', 'name': ''}
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 6. Model Performance — Load and Evaluate

In [ ]:
import tensorflow as tf
import joblib
from sklearn.metrics import mean_absolute_error, r2_score

try:
    tdee_model  = tf.keras.models.load_model(MODEL_DIR / 'tdee_model.keras', compile=False)
    tdee_scaler = joblib.load(MODEL_DIR / 'tdee_scaler.pkl')
    le_sex      = joblib.load(MODEL_DIR / 'le_sex.pkl')
    le_activity = joblib.load(MODEL_DIR / 'le_activity.pkl')
    print('✅ Models loaded')

    # Test on holdout
    test = nhanes.sample(2000, random_state=99)
    X_test = np.column_stack([
        test['age'], le_sex.transform(test['sex']),
        test['height_cm'], test['weight_kg'],
        test['body_fat_pct'], test['bmi'],
        le_activity.transform(test['activity_level']), test['sleep_h']
    ]).astype('float32')
    X_s = tdee_scaler.transform(X_test)
    y_pred = tdee_model.predict(X_s, verbose=0).flatten()
    y_true = test['tdee'].values

    print(f'TDEE MAE  : {mean_absolute_error(y_true, y_pred):.1f} kcal')
    print(f'TDEE R²   : {r2_score(y_true, y_pred):.4f}')
    print(f'TDEE MAPE : {np.mean(np.abs((y_true - y_pred) / y_true))*100:.2f}%')

except Exception as e:
    print(f'Models not trained yet: {e}')

## 7. Food Macro Treemap

In [ ]:
fig = px.treemap(
    ifct,
    path=['category', 'name'],
    values='calories',
    color='protein_g',
    color_continuous_scale='YlGn',
    title='Indian Foods — Calorie Size, Protein Color (darker = more protein)'
)
fig.update_layout(height=600)
fig.show()

## 8. Training Plot Gallery

In [ ]:
from IPython.display import Image, display
import os

plots = sorted(PLOT_DIR.glob('*.png')) if PLOT_DIR.exists() else []
for p in plots:
    print(f'\n── {p.stem} ──')
    display(Image(str(p), width=800))